## TUGAS MANDIRI - Debi Aprilia - 2320506038

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun **dashboard pemantauan pesanan real-time** untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (*prototype*) pipeline streaming yang mensimulasikan kebutuhan ini.

### Instruksi Pengerjaan

Buat notebook baru **`Tugas10_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession` (jangan lupa `spark.conf.set("spark.sql.shuffle.partitions", "4")`), lalu kerjakan bagian **A sampai E** berikut:

---

### Persiapan

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import col, sum as spark_sum, count, window
import os
import shutil
import time
import pandas as pd
from datetime import datetime

# Inisialisasi SparkSession dan set partisi shuffle menjadi 4 agar proses lebih cepat
spark = SparkSession.builder.appName("TugasMandiri10").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

26/11/09 00:19:33 WARN Utils: Your hostname, deb-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/09 00:19:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 00:19:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

In [2]:
# 1. Menyiapkan folder simulasi
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

# 2. Mendefinisikan skema secara eksplisit
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# 3. Mendefinisikan readStream
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .load(folder_tugas)

**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.


In [3]:
# Menambahkan kolom total_harga
df_stream_harga = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi
agregasi_cabang = df_stream_harga.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Memulai streaming query
query1 = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.

In [4]:
# Agregasi window waktu 2 menit
agregasi_waktu = df_stream_harga.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

# Memulai streaming query kedua
query2 = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.

In [5]:
#Untuk mengambil waktu saat ini di komputer, lalu diubah ke teks (string)
waktu_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Batch 1
pd.DataFrame({
    "pesanan_id": ["P1", "P2"],
    "nama_cabang": ["Cabang A", "Cabang B"],
    "nama_menu": ["Kopi Susu", "Americano"],
    "qty": [2, 1],
    "harga_satuan": [20000, 15000],
    "waktu_pesan": [waktu_now, waktu_now]
}).to_csv(f"{folder_tugas}/batch1.csv", index=False)

print("Batch 1 dikirim...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang").show()

# Batch 2 
pd.DataFrame({
    "pesanan_id": ["P3", "P4"],
    "nama_cabang": ["Cabang A", "Cabang C"],
    "nama_menu": ["Latte", "Espresso"],
    "qty": [1, 2],
    "harga_satuan": [25000, 10000],
    "waktu_pesan": [waktu_now, waktu_now]
}).to_csv(f"{folder_tugas}/batch2.csv", index=False)

print("Batch 2 dikirim...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang").show()

# Batch 3
pd.DataFrame({
    "pesanan_id": ["P5"],
    "nama_cabang": ["Cabang B"],
    "nama_menu": ["Kopi Susu"],
    "qty": [3],
    "harga_satuan": [20000],
    "waktu_pesan": [waktu_now]
}).to_csv(f"{folder_tugas}/batch3.csv", index=False)

print("Batch 3 dikirim...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang").show()

Batch 1 dikirim...


[Stage 0:>                                                          (0 + 1) / 1]

+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
+-----------+----------------+--------------+

Batch 2 dikirim...


+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Cabang B|           15000|             1|
|   Cabang C|           20000|             1|
|   Cabang A|           65000|             2|
+-----------+----------------+--------------+

Batch 3 dikirim...


+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Cabang B|           75000|             2|
|   Cabang C|           20000|             1|
|   Cabang A|           65000|             2|
+-----------+----------------+--------------+



**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

In [6]:
# Membersihkan folder output sebelumnya jika ada
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

# Memulai query ketiga untuk menyimpan raw file
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

# Batch 4
pd.DataFrame({
    "pesanan_id": ["P6"],
    "nama_cabang": ["Cabang C"],
    "nama_menu": ["Matcha"],
    "qty": [2],
    "harga_satuan": [20000],
    "waktu_pesan": [waktu_now]
}).to_csv(f"{folder_tugas}/batch4.csv", index=False)

print("Batch 4 dikirim ke Parquet...")
time.sleep(8)

# Memverifikasi file Parquet bisa dibaca sebagai batch
df_baca_parquet = spark.read.parquet("tugas10_output")
df_baca_parquet.show()

Batch 4 dikirim ke Parquet...


+----------+-----------+---------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+---------+---+------------+-------------------+
|        P1|   Cabang A|Kopi Susu|  2|       20000|2026-11-09 00:20:00|
|        P2|   Cabang B|Americano|  1|       15000|2026-11-09 00:20:00|
|        P3|   Cabang A|    Latte|  1|       25000|2026-11-09 00:20:00|
|        P4|   Cabang C| Espresso|  2|       10000|2026-11-09 00:20:00|
|        P5|   Cabang B|Kopi Susu|  3|       20000|2026-11-09 00:20:00|
|        P6|   Cabang C|   Matcha|  2|       20000|2026-11-09 00:20:00|
+----------+-----------+---------+---+------------+-------------------+



In [7]:
#Untuk membuktikan berkas parquetnya benar benar tersimpan
!ls tugas10_output

part-00000-94fc8fcc-77a6-4dd8-b0fd-b2669ee8969c-c000.snappy.parquet
part-00001-28963bf3-b7ea-4346-bdd2-82809b1f3986-c000.snappy.parquet
_spark_metadata


### Tutup

In [8]:
query1.stop()
query2.stop()
query3.stop()
spark.stop()
print("Seluruh streaming query berhasil dihentikan!")

Seluruh streaming query berhasil dihentikan!
